# Vision Language Chat (VLM)

## Task Description
Chat with an image using a Vision Language Model (VLM).

In this notebook, we will:
1. Install necessary libraries.
2. Load an image from a URL.
3. Load **Phi-3-Vision**, a state-of-the-art small VLM optimized for T4 GPUs.
4. Ask questions about the image and get natural language responses.


## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source models and is optimized to run on the Google Colab Free Tier.

## 1. Installation
We need `transformers`, `torch`, `einops`, and `accelerate`.

In [ ]:
!pip install -q transformers>=4.40 accelerate bitsandbytes matplotlib torch einops

## 2. Imports
Importing the necessary Python modules.

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor
from PIL import Image
import requests
import torch

## 3. Load Data
We will load a sample image from the COCO dataset.

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
display(image)
print("Image loaded.")

## 4. Data Cleaning
Ensure image format.

In [ ]:
if image.mode != 'RGB':
    image = image.convert('RGB')
print("Image ready.")

## 5. Load Model
Load `microsoft/Phi-3-vision-128k-instruct`.

In [ ]:
model_id = "microsoft/Phi-3-vision-128k-instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    trust_remote_code=True,
    torch_dtype="auto",
    _attn_implementation='eager'
)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

print("Phi-3-Vision loaded.")

## 6. Inference


In [ ]:
question = "Describe this image in detail."
messages = [
    {"role": "user", "content": f"<|image_1|>\n{question}"}
]

prompt = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(prompt, [image], return_tensors="pt").to("cuda")

generation_args = {
    "max_new_tokens": 500,
    "temperature": 0.0,
    "do_sample": False,
}

generate_ids = model.generate(**inputs, eos_token_id=processor.tokenizer.eos_token_id, **generation_args)

# Remove input tokens
generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
response = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]

generated_text = response
print(f"Question: {question}")
print(f"Answer: {generated_text}")

## 7. Save Results


In [ ]:
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'frames' in locals():
        f.write("Video generated.")
    elif 'image' in locals():
        f.write("Image generated.")
    elif 'text' in locals():
        f.write(str(text))
    else:
        f.write("Task completed.")
print('Results saved to results.txt')


## 8. Advanced Usage


In [ ]:
# Example: Load a different model
# model_name = "..."
# classifier = pipeline(..., model=model_name)
# results = classifier(your_input)
print("Try modifying the code above to use a different model!")